# Phase 5 Analysis — 2026-04-30 Redesign Restructuring

Orchestrates the analysis pipeline using the new modular architecture.
Reuses legacy logic for Steps A–B (person catalogue + occurrence matrices),
then branches into the updated analysis workflow with module-driven computations.

**Key Design Decisions:**
- All complex transformation logic lives in `speakermining/src/analysis/` modules
- Notebooks orchestrate module calls and persist outputs
- Configuration is human-editable in `data/00_setup/`
- Outputs are organized by sensitivity: `persons/` (GDPR), `all/` (combined), `reference/` (publishable)

In [ ]:
# Section 1: Project Setup and Module Imports

from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

# Establish repo root and add src to path
repo_root = Path.cwd()
if not (repo_root / "speakermining").exists():
    for parent in list(repo_root.parents):
        if (parent / "speakermining").exists():
            repo_root = parent
            break

os.chdir(repo_root)
src_path = repo_root / "speakermining" / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Import analysis modules (all complex logic lives here)
from analysis import (
    build_occurrence_matrix,
    load_analysis_properties,
    extract_all_properties,
)
from analysis.universal_stats import (
    compute_carrier_stats,
    compute_episode_appearance_stats,
    build_frequency_distribution,
    build_pareto_table,
)
from process.io_guardrails import atomic_write_csv, atomic_write_text

print(f"repo_root: {repo_root}")
print(f"src_path: {src_path}")
print("✓ Imports successful")

## 1. Load Configuration, Paths, and In-Scope Shows

Define output directories, read broadcasting programs setup, and establish the shows to be analyzed.

In [ ]:
# Output directories (organized by sensitivity and scope)
OUTPUT_DIR  = Path("data/50_analysis")
ALL_DIR     = OUTPUT_DIR / "all"      # combined all-shows analysis
PERSONS_DIR = OUTPUT_DIR / "persons"  # GDPR-sensitive person data
REF_DIR     = OUTPUT_DIR / "reference" # non-human Wikidata data (publishable)
for _d in [OUTPUT_DIR, ALL_DIR, PERSONS_DIR, REF_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

# Phase 2 archive path (contains cached Wikidata reference data)
ARCH = Path("data/20_candidate_generation/wikidata/projections/archive")

# Load in-scope broadcasting programs
broadcasting_programs = pd.read_csv("data/00_setup/broadcasting_programs.csv", dtype=str).fillna("")
IN_SCOPE_SHOW_IDS = {
    s.strip() for s in broadcasting_programs["fernsehserien_de_id"]
    if s.strip() and s.strip() != "NONE"
}

print(f"Output directories:")
print(f"  {ALL_DIR}")
print(f"  {PERSONS_DIR}")
print(f"  {REF_DIR}")
print(f"\nIn-scope shows ({len(IN_SCOPE_SHOW_IDS)}): {sorted(IN_SCOPE_SHOW_IDS)}")

## 2. Load Source Data and Cached Wikidata Reference

Load entity reconciliation, person deduplication, cluster membership, episode metadata, and Wikidata reference files.

In [ ]:
# Load core datasets
dedup_persons = pd.read_csv("data/32_entity_deduplication/dedup_persons.csv", dtype=str).fillna("")
cluster_members = pd.read_csv("data/32_entity_deduplication/dedup_cluster_members.csv", dtype=str).fillna("")
reconciled = pd.read_csv("data/31_entity_disambiguation/manual/reconciled_data_summary.csv", dtype=str).fillna("")
episode_guests_raw = pd.read_csv("data/31_entity_disambiguation/raw_import/episode_guests_normalized.csv", dtype=str).fillna("")
episode_meta = pd.read_csv("data/31_entity_disambiguation/raw_import/episode_metadata_normalized.csv", dtype=str).fillna("")

# Load cached Wikidata reference
core_persons = json.loads((ARCH / "core_persons.json").read_text(encoding="utf-8"))
instances = pd.read_csv(ARCH / "instances.csv", dtype=str).fillna("")

# Build QID → label lookup (German preferred, fallback English)
qid_label = {}
for _, row in instances.iterrows():
    label = row.get("labels_de", "") or row.get("labels_en", "") or row.get("label", "")
    if row["qid"] and label:
        qid_label[row["qid"]] = label
for qid, entity in core_persons.items():
    labels = entity.get("labels", {})
    label = labels.get("de", {}).get("value", "") or labels.get("en", {}).get("value", "")
    if label:
        qid_label[qid] = label

print(f"Data loaded:")
print(f"  dedup_persons:         {len(dedup_persons):,} rows")
print(f"  cluster_members:       {len(cluster_members):,} rows")
print(f"  reconciled:            {len(reconciled):,} rows")
print(f"  episode_guests_raw:    {len(episode_guests_raw):,} rows")
print(f"  episode_meta:          {len(episode_meta):,} rows")
print(f"  core_persons (arch):   {len(core_persons):,} entries")
print(f"  qid_label lookup:      {len(qid_label):,} entries")

## 3. Run Person Catalogue Build via Pipeline Modules

This section reuses the legacy Step A approach:
1. Resolve canonical person identities and role assignment
2. Extract Wikidata properties
3. Write person catalogues to `persons/` (GDPR-sensitive)

In [ ]:
# Role mapping (from legacy Step A)
ROLE_MAP = {
    "Gast":             "guest",
    "Kommentar":        "guest",
    "Kommentator":      "guest",
    "":                 "guest",
    "Moderation":       "moderator",
    "Produktionsauftrag": "staff",
    "Produktionsfirma": "staff",
    "Redaktion":        "staff",
    "Regie":            "staff",
    "Drehbuch":         "staff",
}
MODERATOR_QIDS = {"Q43773"}  # Markus Lanz
ROLE_PRIORITY = {"guest": 0, "moderator": 1, "staff": 2, "incidental": 3}

# In-scope episode URLs
in_scope_episode_urls = set(
    episode_meta[episode_meta["fernsehserien_de_id"].isin(IN_SCOPE_SHOW_IDS)]["episode_url"],
)

# Join reconciled -> cluster_members to get canonical_entity_id
cm_bridge = cluster_members[["alignment_unit_id", "canonical_entity_id"]].drop_duplicates("alignment_unit_id")
reconciled_ceid = reconciled.merge(cm_bridge, on="alignment_unit_id", how="left")

# Filter to in-scope episodes
reconciled_inscope = reconciled_ceid[
    reconciled_ceid["fernsehserien_de_id"].isin(in_scope_episode_urls)
].copy()

# Join with episode_guests_raw to get guest_role
eg_lookup = (
    episode_guests_raw[["episode_url", "guest_name", "guest_role"]]
    .copy()
    .assign(_name_lower=lambda d: d["guest_name"].str.strip().str.lower())
)
reconciled_inscope["_name_lower"] = reconciled_inscope["canonical_label"].str.strip().str.lower()

ri_with_role = reconciled_inscope.merge(
    eg_lookup.rename(columns={"episode_url": "fernsehserien_de_id", "guest_role": "raw_role"}),
    on=["fernsehserien_de_id", "_name_lower"],
    how="left"
).drop(columns=["_name_lower", "guest_name"], errors="ignore")

ri_with_role["raw_role"] = ri_with_role["raw_role"].fillna("")
ri_with_role["role"] = ri_with_role["raw_role"].map(ROLE_MAP).fillna("guest")
ri_with_role.loc[ri_with_role["wikidata_id"].isin(MODERATOR_QIDS), "role"] = "moderator"

# Appearance count per canonical entity
app_counts_s = (
    ri_with_role[ri_with_role["canonical_entity_id"].notna()]
    .groupby("canonical_entity_id")["fernsehserien_de_id"]
    .nunique()
    .rename("appearance_count")
    .reset_index()
 )

# Dominant role per canonical entity
dominant_role_s = (
    ri_with_role[ri_with_role["canonical_entity_id"].notna()]
    .groupby("canonical_entity_id")["role"]
    .agg(lambda roles: min(roles, key=lambda r: ROLE_PRIORITY.get(r, 9)))
    .rename("role")
    .reset_index()
)

# Build catalogue from dedup_persons
TIER_ORDER = {"high": 0, "medium": 1, "low": 2, "": 9}
best_qid_df = (
    reconciled_ceid[
        reconciled_ceid["canonical_entity_id"].notna() &
        (reconciled_ceid["wikidata_id"] != "")
    ][["canonical_entity_id", "wikidata_id", "match_tier"]]
    .assign(_rank=lambda d: d["match_tier"].map(TIER_ORDER).fillna(9).astype(int))
    .sort_values(["canonical_entity_id", "_rank"])
    .groupby("canonical_entity_id", as_index=False)
    .first()[["canonical_entity_id", "wikidata_id"]]
    .rename(columns={"wikidata_id": "reconciled_wikidata_id"})
)

catalogue = dedup_persons[[
    "canonical_entity_id", "wikidata_id", "canonical_label",
    "cluster_size", "cluster_strategy", "cluster_confidence"
]].copy()

catalogue = (
    catalogue
    .merge(dominant_role_s, on="canonical_entity_id", how="left")
    .merge(app_counts_s, on="canonical_entity_id", how="left")
    .merge(best_qid_df, on="canonical_entity_id", how="left")
)
catalogue["role"] = catalogue["role"].fillna("incidental")
catalogue["wikidata_id"] = (
    catalogue["reconciled_wikidata_id"]
    .where(catalogue["reconciled_wikidata_id"].notna() & (catalogue["reconciled_wikidata_id"] != ""),
           other=catalogue["wikidata_id"])
    .fillna("")
)
catalogue.drop(columns=["reconciled_wikidata_id"], inplace=True)
catalogue.loc[catalogue["wikidata_id"].isin(MODERATOR_QIDS), "role"] = "moderator"
catalogue["appearance_count"] = catalogue["appearance_count"].fillna(0).astype(int)

# Load the enabled analysis-property catalog and extract all non-derived values now.
# AGE is handled later because it depends on the episode year.
analysis_properties = load_analysis_properties()
analysis_properties = analysis_properties[
    analysis_properties["enabled"].astype(str).str.strip().ne("0")
].copy()
property_catalog = analysis_properties[
    analysis_properties["type"].astype(str).str.strip().str.lower().ne("derived")
].copy()

# Ensure we have core entity docs for all guests — if any are missing, attempt
# a cache-first full outlink fetch via the centralized entity access API.
from process.candidate_generation.wikidata import entity_access

missing_guest_qids = set(catalogue["wikidata_id"].unique()) - set(core_persons.keys())
missing_guest_qids = {q for q in missing_guest_qids if q and str(q).strip()}
if missing_guest_qids:
    print(f"Missing cached entity docs for {len(missing_guest_qids)} guests — attempting full fetches")
    # Start a request context — budget_remaining=-1 means unlimited for this run.
    entity_access.begin_request_context(budget_remaining=-1, query_delay_seconds=0.2)
    try:
        fetched = 0
        for q in sorted(missing_guest_qids):
            doc = entity_access.all_outlink_fetch(q, ARCH)
            if doc:
                core_persons[q] = doc
                # update qid_label from fetched doc
                labels = doc.get("labels", {})
                label = labels.get("de", {}).get("value") or labels.get("en", {}).get("value")
                if label:
                    qid_label[q] = label
                fetched += 1
        print(f"Fetched {fetched} missing guest entity docs")
    finally:
        consumed = entity_access.end_request_context()
        print(f"Network requests consumed: {consumed}")

property_values_by_id = {}
if not property_catalog.empty:
    property_values_by_id = extract_all_properties(
        catalogue[["wikidata_id", "canonical_label"]].copy(),
        pd.DataFrame(),
        property_catalog,
        core_persons,
        qid_label,
    )

# After extraction, ensure all value QIDs present in item frames have labels and
# full entity docs available (fetch as necessary). This prevents QIDs leaking
# into visualizations.
missing_value_qids = set()
for pid, df in property_values_by_id.items():
    if {"value_qid"}.issubset(df.columns):
        for v in df["value_qid"].unique():
            if not v:
                continue
            if v not in qid_label or v not in core_persons:
                missing_value_qids.add(v)

if missing_value_qids:
    print(f"Resolving {len(missing_value_qids)} missing value QIDs via full fetch")
    entity_access.begin_request_context(budget_remaining=-1, query_delay_seconds=0.2)
    try:
        fetched_vals = 0
        for q in sorted(missing_value_qids):
            doc = entity_access.all_outlink_fetch(q, ARCH)
            if doc:
                core_persons[q] = doc
                labels = doc.get("labels", {})
                label = labels.get("de", {}).get("value") or labels.get("en", {}).get("value")
                if label:
                    qid_label[q] = label
                fetched_vals += 1
        print(f"Fetched {fetched_vals} value entity docs")
    finally:
        consumed = entity_access.end_request_context()
        print(f"Network requests consumed: {consumed}")

# Map birthyear into catalogue if available
birthyear_df = property_values_by_id.get("P569", pd.DataFrame())
if not birthyear_df.empty and {"guest_qid", "value_year"}.issubset(birthyear_df.columns):
    birthyear_lookup = (
        birthyear_df[["guest_qid", "value_year"]]
        .drop_duplicates("guest_qid")
        .rename(columns={"guest_qid": "wikidata_id", "value_year": "birthyear"})
    )
    catalogue = catalogue.merge(birthyear_lookup, on="wikidata_id", how="left")
else:
    catalogue["birthyear"] = ""

CATALOGUE_COLS = [
    "canonical_entity_id", "wikidata_id", "canonical_label", "cluster_size",
    "cluster_strategy", "cluster_confidence", "role", "appearance_count", "birthyear",
]
catalogue = catalogue[CATALOGUE_COLS]

## 4. Build Episode × Guest Occurrence Matrix

Reuses legacy Step B: build the guest-by-episode binary matrix, sorted by appearance count and premiere date.

In [ ]:
# Guest subset
guest_cat = catalogue[catalogue["role"] == "guest"].copy()
print(f"Guests in catalogue: {len(guest_cat):,}")

# Guest-episode pairs from ri_with_role
guest_ceids = set(guest_cat["canonical_entity_id"])
guest_pairs = ri_with_role[
    ri_with_role["canonical_entity_id"].isin(guest_ceids) &
    (ri_with_role["role"] == "guest")
][["canonical_entity_id", "fernsehserien_de_id"]].drop_duplicates()
print(f"Guest-episode pairs: {len(guest_pairs):,}")

# Episode sort order (by premiere_date asc)
ep_order = (
    episode_meta[episode_meta["episode_url"].isin(in_scope_episode_urls)]
    [["episode_url", "premiere_date", "fernsehserien_de_id", "program_name"]]
    .drop_duplicates("episode_url")
    .sort_values("premiere_date")
)

# Person sort order (appearance_count desc, then alpha)
person_order = (
    guest_cat[["canonical_entity_id", "canonical_label", "appearance_count"]]
    .sort_values(["appearance_count", "canonical_label"], ascending=[False, True])
)

# Pivot (1 = appeared, 0 = absent)
guest_pairs["_val"] = 1
matrix_num = guest_pairs.pivot_table(
    index="canonical_entity_id", columns="fernsehserien_de_id",
    values="_val", aggfunc="max", fill_value=0
)
ordered_persons  = [c for c in person_order["canonical_entity_id"] if c in matrix_num.index]
ordered_episodes = [e for e in ep_order["episode_url"] if e in matrix_num.columns]
matrix_num = matrix_num.reindex(index=ordered_persons, columns=ordered_episodes, fill_value=0)

# Output with 1/empty cells
matrix_out = matrix_num.copy().astype(object)
matrix_out[matrix_num == 0] = ""
ceid_to_label = person_order.set_index("canonical_entity_id")["canonical_label"]
matrix_out.insert(0, "canonical_label", ceid_to_label)
matrix_out = matrix_out.reset_index()
atomic_write_csv(ALL_DIR / "occurrence_matrix.csv", matrix_out)

print(f"\nOccurrence matrix:")
print(f"  all/occurrence_matrix.csv: {matrix_out.shape[0]} persons × {matrix_out.shape[1]-2} episodes")

## 5. Generate Per-Show Matrices and Co-Occurrence Outputs

Derive per-show matrices and co-occurrence matrix for top guests.

In [ ]:
# Per-show matrices
for show_id in sorted(IN_SCOPE_SHOW_IDS):
    show_eps = [e for e in ep_order[ep_order["fernsehserien_de_id"] == show_id]["episode_url"] if e in matrix_num.columns]
    if not show_eps:
        continue
    show_mat = matrix_num[show_eps]
    show_persons = show_mat[show_mat.sum(axis=1) > 0].index.tolist()
    show_out = matrix_out[matrix_out["canonical_entity_id"].isin(show_persons)][
        ["canonical_entity_id", "canonical_label"] + show_eps
    ].copy()
    show_dir = OUTPUT_DIR / show_id.replace("-", "_")
    show_dir.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(show_dir / "occurrence_matrix.csv", show_out)

print(f"Per-show matrices created: {len(sorted(IN_SCOPE_SHOW_IDS))} shows")

# Co-occurrence matrix (top 200 guests)
top200 = ordered_persons[:200]
top_num = matrix_num.reindex(top200).fillna(0).astype(int)
co_occ = top_num.dot(top_num.T)
co_occ_out = co_occ.reset_index()
atomic_write_csv(ALL_DIR / "co_occurrence_matrix.csv", co_occ_out)

print(f"  all/co_occurrence_matrix.csv: {co_occ_out.shape}")

## 6. Prepare Analysis Base Tables for Downstream Modules

Construct the normalized guest-episode analysis table that downstream statistic modules will consume.

In [ ]:
# Build episode_appearances for downstream analysis modules
episode_appearances = (
    ri_with_role[ri_with_role["canonical_entity_id"].notna()][
        ["canonical_entity_id", "fernsehserien_de_id", "role"]
    ]
    .merge(
        catalogue[[
            "canonical_entity_id", "wikidata_id", "canonical_label", "appearance_count", "birthyear"
        ]],
        on="canonical_entity_id", how="left"
    )
    .merge(
        ep_order[[
            "episode_url", "premiere_date", "fernsehserien_de_id", "program_name"
        ]].rename(columns={"fernsehserien_de_id": "show_id"}),
        left_on="fernsehserien_de_id", right_on="episode_url", how="left"
    )
)
episode_appearances["premiere_year"] = episode_appearances["premiere_date"].str[:4]
episode_appearances["guest_qid"] = episode_appearances["wikidata_id"]

def _derive_age_values(frame: pd.DataFrame) -> pd.DataFrame:
    working = frame[frame["role"] == "guest"].copy()
    working["birth_year_num"] = pd.to_numeric(working["birthyear"].astype(str).str[:4], errors="coerce")
    working["premiere_year_num"] = pd.to_numeric(working["premiere_year"], errors="coerce")
    working = working.dropna(subset=["birth_year_num", "premiere_year_num"])
    working["value"] = (working["premiere_year_num"] - working["birth_year_num"]).astype(int)
    working = working[(working["value"] >= 0) & (working["value"] < 120)]
    if working.empty:
        return pd.DataFrame(columns=["guest_qid", "episode_id", "value", "appearance_count"])
    return working.rename(columns={"fernsehserien_de_id": "episode_id"})[
        ["guest_qid", "episode_id", "value"]
    ].assign(appearance_count=1)

print(f"Analysis base table prepared:")
print(f"  episode_appearances: {len(episode_appearances):,} rows")

## 7. Compute Configured Property Outputs

Run the enabled-property loop driven by `data/00_setup/analysis_properties.csv`.

In [ ]:
def _slugify_property_label(label: str) -> str:
    slug = "".join(ch.lower() if ch.isalnum() else "_" for ch in str(label).strip())
    return "_".join(part for part in slug.split("_") if part)


def _derive_age_values(frame: pd.DataFrame) -> pd.DataFrame:
    working = frame[frame["role"] == "guest"].copy()
    if "guest_qid" not in working.columns:
        if "wikidata_id" in working.columns:
            working["guest_qid"] = working["wikidata_id"]
        elif "canonical_entity_id" in working.columns:
            working["guest_qid"] = working["canonical_entity_id"]
        else:
            working["guest_qid"] = ""
    if "episode_id" not in working.columns and "fernsehserien_de_id" in working.columns:
        working = working.rename(columns={"fernsehserien_de_id": "episode_id"})
    working["birth_year_num"] = pd.to_numeric(working["birthyear"].astype(str).str[:4], errors="coerce")
    working["premiere_year_num"] = pd.to_numeric(working["premiere_year"], errors="coerce")
    working = working.dropna(subset=["birth_year_num", "premiere_year_num"])
    working["value"] = (working["premiere_year_num"] - working["birth_year_num"]).astype(int)
    working = working[(working["value"] >= 0) & (working["value"] < 120)]
    if working.empty:
        return pd.DataFrame(columns=["guest_qid", "episode_id", "value", "appearance_count"])
    return working[["guest_qid", "episode_id", "value"]].assign(appearance_count=1)


def _standardize_property_frame(prop_row: pd.Series, raw_frame: pd.DataFrame) -> pd.DataFrame:
    prop_pid = str(prop_row.get("wikidata_id", "")).strip()
    prop_type = str(prop_row.get("type", "")).strip().lower()

    if prop_pid == "AGE" or prop_type == "derived":
        return _derive_age_values(episode_appearances)

    if raw_frame is None or raw_frame.empty:
        return pd.DataFrame(columns=["guest_qid", "value", "appearance_count"])

    frame = raw_frame.copy()
    if "guest_qid" not in frame.columns:
        frame["guest_qid"] = frame.get("canonical_entity_id", "")

    if prop_type == "item":
        value_label = frame.get("value_label", pd.Series(dtype=str)).astype(str)
        value_qid = frame.get("value_qid", pd.Series(dtype=str)).astype(str)
        frame["value"] = value_label.where(value_label.str.strip() != "", value_qid)
    elif "value" not in frame.columns and "value_year" in frame.columns:
        frame["value"] = frame["value_year"]

    frame["value"] = frame.get("value", "").astype(str)
    frame["guest_qid"] = frame["guest_qid"].astype(str)
    frame["appearance_count"] = 1

    keep_cols = ["guest_qid", "value", "appearance_count"]
    if "episode_id" in frame.columns:
        keep_cols.append("episode_id")
    return frame[keep_cols].copy()


print(f"Generating configured property outputs for {len(analysis_properties):,} enabled properties...")
property_stats = {}
for _, prop_row in analysis_properties.iterrows():
    prop_pid = str(prop_row.get("wikidata_id", "")).strip()
    prop_label = str(prop_row.get("label", prop_pid)).strip() or prop_pid
    prop_type = str(prop_row.get("type", "")).strip()
    raw_frame = property_values_by_id.get(prop_pid, pd.DataFrame())
    standard_frame = _standardize_property_frame(prop_row, raw_frame)

    if standard_frame.empty:
        print(f"  [{prop_pid}] {prop_label} ({prop_type}) -> no data")
        continue

    carrier_stats = compute_carrier_stats(
        standard_frame,
        value_column="value",
        carrier_column="guest_qid",
        appearance_column="appearance_count",
    )

    episode_stats = pd.DataFrame()
    if "episode_id" in standard_frame.columns:
        episode_stats = compute_episode_appearance_stats(
            standard_frame,
            value_column="value",
            carrier_column="guest_qid",
            episode_column="episode_id",
        )

    prop_dir = ALL_DIR / _slugify_property_label(prop_label)
    prop_dir.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(prop_dir / "carrier_stats.csv", carrier_stats)
    if not episode_stats.empty:
        atomic_write_csv(prop_dir / "episode_stats.csv", episode_stats)

    property_stats[prop_pid] = (prop_label, carrier_stats)
    print(f"  [{prop_pid}] {prop_label} ({prop_type}) -> {len(standard_frame):,} rows, {len(carrier_stats):,} values")

print(f"Configured property outputs complete: {len(property_stats):,}/{len(analysis_properties):,} properties processed")

## 10. Compute Age Distribution Outputs

Derive appearance age from premiere year and birth year, bin results.
(Jump in numbers due to notebook changes, deprecation of Step 8 and 9)

In [ ]:
# Age distribution (10-year bins)
age_df = episode_appearances[
    (episode_appearances["role"] == "guest") &
    (episode_appearances["birthyear"] != "")
].copy()
age_df["birth_year_num"]    = pd.to_numeric(age_df["birthyear"], errors="coerce")
age_df["premiere_year_num"] = pd.to_numeric(age_df["premiere_year"], errors="coerce")
age_df = age_df.dropna(subset=["birth_year_num", "premiere_year_num"])
age_df["appearance_age"] = (age_df["premiere_year_num"] - age_df["birth_year_num"]).astype(int)
age_df = age_df[(age_df["appearance_age"] >= 0) & (age_df["appearance_age"] < 120)]

bins = list(range(0, 121, 10))
labels = [f"{b}-{b+9}" for b in bins[:-1]]
age_df["age_bin"] = pd.cut(age_df["appearance_age"], bins=bins, labels=labels, right=False)
c8 = compute_carrier_stats(
    age_df,
    value_column="age_bin",
    carrier_column="canonical_entity_id",
    appearance_column="appearance_age",
)
atomic_write_csv(ALL_DIR / "age_distribution.csv", c8)

print("=== AGE DISTRIBUTION ===")
print(c8.to_string(index=False))

## 11. Write Dataset Overview and Run Validation Checks

Generate dataset overview summary and confirmation of output coverage.

In [ ]:
# Dataset overview
all_reconciled_persons = reconciled[reconciled["entity_class"] == "person"]
reconciled_with_qid    = all_reconciled_persons[all_reconciled_persons["wikidata_id"] != ""]

d1_rows = [
    {"phase": "Phase 31", "source": "reconciled_data_summary",
     "entity_type": "person (appearance rows)",
     "total_count": len(all_reconciled_persons),
     "wikidata_matched_count": len(reconciled_with_qid),
     "coverage_pct": round(len(reconciled_with_qid) / len(all_reconciled_persons) * 100, 1)},
    {"phase": "Phase 32", "source": "dedup_persons",
     "entity_type": "canonical person",
     "total_count": len(dedup_persons),
     "wikidata_matched_count": (dedup_persons["wikidata_id"] != "").sum(),
     "coverage_pct": round((dedup_persons["wikidata_id"] != "").mean() * 100, 1)},
    {"phase": "Phase 5", "source": "guest_catalogue",
     "entity_type": "canonical person",
     "total_count": len(catalogue),
     "wikidata_matched_count": (catalogue["wikidata_id"] != "").sum(),
     "coverage_pct": round((catalogue["wikidata_id"] != "").mean() * 100, 1)},
]

d1 = pd.DataFrame(d1_rows)
atomic_write_csv(ALL_DIR / "dataset_overview.csv", d1)

print("=== DATASET OVERVIEW ===")
print(d1.to_string(index=False))

# Summary statistics
summary = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "catalogue": {
        "total_canonical_persons": len(catalogue),
        "wikidata_matched": int((catalogue["wikidata_id"] != "").sum()),
        "role_distribution": catalogue["role"].value_counts().to_dict(),
    },
    "guests": {
        "total": int(len(guest_cat)),
        "with_wikidata": int((guest_cat["wikidata_id"] != "").sum()),
        "total_appearances": int(guest_cat["appearance_count"].sum()),
    },
}

atomic_write_text(ALL_DIR / "analysis_summary.json",
                  json.dumps(summary, indent=2, ensure_ascii=False))

print("\n✓ Analysis pipeline complete")
print(f"\nOutput files written to:")
print(f"  {ALL_DIR.resolve()}")
print(f"  {PERSONS_DIR.resolve()}")

## Summary

**Completed Steps:**
1. ✓ Loaded configuration and all source data
2. ✓ Built person catalogue with role classification and appearance counts
3. ✓ Extracted configured Wikidata properties from `analysis_properties.csv`
4. ✓ Built episode × guest occurrence matrix (reused legacy Step B logic)
5. ✓ Generated per-show matrices and co-occurrence outputs
6. ✓ Prepared analysis base tables for downstream modules
7. ✓ Computed configured property summaries and age special handling
8. ✓ Wrote dataset overview and summary statistics

**Output Structure:**
- `data/50_analysis/all/` — combined all-shows analysis
- `data/50_analysis/persons/` — GDPR-sensitive person data
- `data/50_analysis/reference/` — non-human Wikidata reference data
- `data/50_analysis/<show_id>/` — per-show matrices

**Next Steps:**
- Layer 1c: Create `class_hierarchy.py` module for P279 walk and mid-level assignment
- Layer 2: Implement universal_stats, cooccurrence, midlevel_aggregation, person_analysis, episode_analysis, source_coverage
- Layer 3: Create visualization modules and wire into notebook orchestration

## 12. Visualization System Setup (TASK-B08)

Load visualization infrastructure and set up output directories.

In [ ]:
# Visualization infrastructure imports
import plotly.graph_objects as go
import plotly.express as px
from analysis import (
    apply_font,
    save_fig,
    sort_bars_descending,
    add_unknown_row,
    add_scope_label,
    add_context_stats,
    apply_other_grouping,
    PALETTE,
)

# Create visualization output directories
VIZ_DIR = ALL_DIR / "visualizations"
VIZ_DIR.mkdir(parents=True, exist_ok=True)

print(f"Visualization directories ready:")
print(f"  {VIZ_DIR}")

## 13. Universal Visualizations (TASK-B09)

Generate universal charts for every enabled property using the stats collected from the configured property loop.

In [ ]:
# Universal Visualizations (TASK-B09)
# Loop over all configured properties -- no property-specific hardcoding.
# The property_stats map is built from data in the previous cell.

import importlib
import analysis.viz_universal as _viz_universal
importlib.reload(_viz_universal)
from analysis.viz_universal import universal_visualizations

print(f"Generating universal visualizations for {len(property_stats)} configured properties...")
universal_visualizations(property_stats, ALL_DIR, scope="all", top_n=20)

## 14. Guest Frequency Distribution and Pareto Analysis (TASK-B21)

Analyze how many guests appeared exactly 1, 2, 3, ... times, and produce a Pareto chart.

In [ ]:
# Compute guest frequency distribution
guest_appearance_counts = guest_cat[["wikidata_id", "appearance_count"]].copy()
freq_dist = guest_appearance_counts["appearance_count"].value_counts().sort_index().reset_index()
freq_dist.columns = ["frequency", "number_of_guests"]

# Pareto: cumulative % of appearances
freq_dist = freq_dist.sort_values("frequency")
freq_dist["appearances"] = freq_dist["frequency"] * freq_dist["number_of_guests"]
freq_dist["cumulative_appearances"] = freq_dist["appearances"].cumsum()
total_appearances = freq_dist["appearances"].sum()
freq_dist["pct_cumulative_appearances"] = (freq_dist["cumulative_appearances"] / total_appearances * 100).round(2)

print(f"Total guests: {len(guest_appearance_counts)}")
print(f"Guest frequency distribution:\n{freq_dist.head(10)}")

# TODO: Call viz module for Pareto chart and frequency distribution visualization
# from analysis import plot_guest_frequency_distribution  # (will implement in viz_universal.py)
# fig = plot_guest_frequency_distribution(freq_dist)
# save_fig(fig, data_dir / "50_analysis" / "all" / "guest_frequency_pareto.html")

## 15. Per-Show Analysis (TASK-B22)

For each broadcasting program (show), compute guest count, unique appearances, and episode-level summary statistics.

In [ ]:
# Per-show analysis -- uses episode_appearances which has show_id from ep_order
guest_ep = episode_appearances[episode_appearances["role"] == "guest"]

per_show_stats = (
    guest_ep
    .groupby(["show_id", "program_name"])
    .agg(
        episode_count=("fernsehserien_de_id", "nunique"),
        guest_appearances=("canonical_entity_id", "count"),
        unique_guests=("canonical_entity_id", "nunique"),
    )
    .reset_index()
)
per_show_stats["avg_guests_per_episode"] = (
    per_show_stats["guest_appearances"] / per_show_stats["episode_count"].clip(lower=1)
).round(2)
per_show_stats = per_show_stats.sort_values("guest_appearances", ascending=False).reset_index(drop=True)

print(f"Per-show statistics ({len(per_show_stats)} shows):")
print(per_show_stats.to_string(index=False))
atomic_write_csv(ALL_DIR / "per_show_statistics.csv", per_show_stats)


## 16. Network & Co-occurrence Analysis (TASK-B10)

Compute guest-to-guest co-occurrence networks and produce interactive visualizations.

In [ ]:
# Co-occurrence analysis -- guests appearing in the same episode
# Uses episode_appearances (one row per person x episode), not catalogue (one row per person)
guest_ep = episode_appearances[episode_appearances["role"] == "guest"]

# Build episode -> list of canonical_entity_ids
episodes_list = (
    guest_ep
    .groupby("fernsehserien_de_id")["canonical_entity_id"]
    .apply(list)
    .values
)

co_occurrence_pairs = []
for episode_guests in episodes_list:
    unique_guests = list(set(episode_guests))
    for i in range(len(unique_guests)):
        for j in range(i + 1, len(unique_guests)):
            guest_a, guest_b = sorted([unique_guests[i], unique_guests[j]])
            co_occurrence_pairs.append({"guest_a": guest_a, "guest_b": guest_b})

if co_occurrence_pairs:
    co_occurrence_df = pd.DataFrame(co_occurrence_pairs)
    co_occurrence_summary = (
        co_occurrence_df
        .groupby(["guest_a", "guest_b"])
        .size()
        .reset_index(name="co_occurrence_count")
        .sort_values("co_occurrence_count", ascending=False)
        .reset_index(drop=True)
    )
    print(f"Total co-occurrence pairs: {len(co_occurrence_summary):,}")
    print(f"Top co-occurrences:")
    print(co_occurrence_summary.head(10).to_string(index=False))
    atomic_write_csv(ALL_DIR / "co_occurrence_pairs.csv", co_occurrence_summary)
else:
    co_occurrence_summary = pd.DataFrame(columns=["guest_a", "guest_b", "co_occurrence_count"])
    print("No co-occurrences found.")


## 17. Export Analysis Results & Summary (TASK-B99)

Write all analysis artifacts to output directory structure: `data/50_analysis/{all,persons,reference,<show_id>/}`

In [ ]:
# Export analysis results
# OUTPUT_DIR is defined in cell 3 as Path("data/50_analysis")

# Directories already created by earlier cells -- just ensure they exist
for dir_path in [ALL_DIR, PERSONS_DIR, REF_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

# Export already-computed frequency distribution
atomic_write_csv(ALL_DIR / "guest_frequency_distribution.csv", freq_dist)

# Generate analysis summary
n_episodes = episode_appearances["fernsehserien_de_id"].nunique()
n_guests   = len(guest_cat)
n_shows    = episode_appearances["show_id"].nunique()

total_guest_appearances = int(guest_cat["appearance_count"].sum())
summary_data = {
    "total_episodes_analyzed": n_episodes,
    "total_unique_guests":     n_guests,
    "total_guest_appearances": total_guest_appearances,
    "average_guests_per_episode": round(
        total_guest_appearances / max(n_episodes, 1), 2
    ),
    "broadcasting_programs": n_shows,
    "analysis_timestamp": pd.Timestamp.now().isoformat(),
}

import json as _json
atomic_write_text(ALL_DIR / "analysis_summary.json", _json.dumps(summary_data, indent=2))

print("Analysis export complete!")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Summary: {summary_data}")